In [14]:
!pip install networkx plotly ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.3 MB/s eta 0:00:00


In [15]:
import networkx as nx
import plotly.graph_objects as go
import random
from networkx.algorithms import community
import ipywidgets as widgets
from IPython.display import display

In [16]:
nodes_slider = widgets.IntSlider(value=15, min=5, max=50, step=1, description='Nodes:')
prob_slider = widgets.FloatSlider(value=0.3, min=0.1, max=1.0, step=0.1, description='Density:')

button = widgets.Button(description="Generate Graph")

display(nodes_slider, prob_slider, button)

IntSlider(value=15, description='Nodes:', max=50, min=5)

FloatSlider(value=0.3, description='Density:', max=1.0, min=0.1)

Button(description='Generate Graph', style=ButtonStyle())

In [29]:
def generate_graph_gradio(n, p):
    import networkx as nx
    import plotly.graph_objects as go
    import random
    from networkx.algorithms import community

    # Create graph
    G = nx.erdos_renyi_graph(n=int(n), p=float(p))
    labels = {i: f'User{i}' for i in G.nodes()}

    # Metrics
    degree_dict = dict(G.degree())
    centrality_dict = nx.betweenness_centrality(G)

    # Communities
    communities = list(community.greedy_modularity_communities(G))
    community_map = {}
    for i, comm in enumerate(communities):
        for node in comm:
            community_map[node] = i

    # Shortest path
    if len(G.nodes()) >= 2:
        node_a, node_b = random.sample(list(G.nodes()), 2)
        try:
            shortest_path = nx.shortest_path(G, source=node_a, target=node_b)
        except:
            shortest_path = []
    else:
        shortest_path = []

    # Influencer
    top_node = max(centrality_dict, key=centrality_dict.get)
    influencer_text = (
    f"🔥 Top Influencer: {labels[top_node]}\n"
    f"📊 Centrality Score: {centrality_dict[top_node]:.4f}\n"
    f"🔗 Shortest Path: {shortest_path}\n"
    f"👥 Communities Detected: {len(communities)}"
)

    # Layout
    pos = nx.spring_layout(G, seed=42)

    # Edges
    edge_x, edge_y = [], []
    path_edges = list(zip(shortest_path, shortest_path[1:]))

    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        mode='lines',
        line=dict(width=2, color='gray'),
        hoverinfo='none'
    )

    # Nodes
    node_colors = []
    node_sizes = []

    for node in G.nodes():
        if node == top_node:
            node_colors.append('red')
            node_sizes.append(30)
        else:
            node_colors.append(community_map[node])
            node_sizes.append(20)

    node_x, node_y, hover_text = [], [], []

    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)

        hover_text.append(
            f"User: {labels[node]}<br>"
            f"Degree: {degree_dict[node]}<br>"
            f"Centrality: {centrality_dict[node]:.3f}<br>"
            f"Community: {community_map[node]}"
        )

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        text=[labels[node] for node in G.nodes()],
        textposition="bottom center",
        hovertext=hover_text,
        hoverinfo='text',
        marker=dict(
            size=node_sizes,
            color=node_colors,
            colorscale='Viridis',
            showscale=True
        )
    )

    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        title='🚀 Smart Network Graph Analyzer',
                        showlegend=False,
                        hovermode='closest'
                    ))

    return fig, influencer_text

In [30]:
path_edges = list(zip(shortest_path, shortest_path[1:]))

edge_x, edge_y = [], []

for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]

    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

    if edge in path_edges or (edge[1], edge[0]) in path_edges:
        color = 'red'
        width = 4
    else:
        color = 'gray'
        width = 1

    fig.add_trace(go.Scatter(
        x=[x0, x1],
        y=[y0, y1],
        mode='lines',
        line=dict(color=color, width=width),
        hoverinfo='none'
    ))

In [31]:
button.on_click(generate_graph)

In [32]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## 🚀 Smart Graph Theory Analyzer")

    with gr.Row():
        n_input = gr.Slider(5, 50, value=15, step=1, label="Number of Nodes")
        p_input = gr.Slider(0.1, 1.0, value=0.3, step=0.1, label="Connection Density")

    btn = gr.Button("Generate Graph")

    graph_output = gr.Plot()
    text_output = gr.Textbox(label="Insights")

    btn.click(
        fn=generate_graph_gradio,
        inputs=[n_input, p_input],
        outputs=[graph_output, text_output]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://69da93ae8058df5a4e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
